# P31 · Phase 2 close-out: benchmark, video run, running service, checksums, segmentation threshold on validation
Run top to bottom (Runtime -> Run all) on a **T4 GPU**. No test split is read anywhere in this notebook.
Sections: 1 setup · 2 checksums · 3 pipeline from the manifest · 4 benchmark (warm-up, p50/p95/p99, throughput, memory)
· 5 video run at the validation operating point · 6 HTTP service (health, analyze, confirm) · 7 segmentation threshold on validation.
Outputs: `04_Integration/2026-11-18_P31_Closeout/`. The Al-Hada clip and the annotated video stay on Drive (owner only), never in the repo.

In [ ]:
# setup
# Who runs what: on a T4 everything runs. Without a GPU (Colab limit) the benchmark is skipped and the rest runs on CPU.
# To run ONLY the benchmark (e.g. a teammate with GPU quota), set ONLY_BENCHMARK = True.
ONLY_BENCHMARK = False
import subprocess, sys
PKG = {'fastapi': 'fastapi', 'uvicorn': 'uvicorn', 'multipart': 'python-multipart', 'onnxruntime': 'onnxruntime-gpu', 'ultralytics': 'ultralytics'}
need = [p for m, p in PKG.items() if subprocess.run([sys.executable, '-c', f'import {m}']).returncode]
if need: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True); print('installed', need)
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, time, hashlib, threading, datetime, random, resource, numpy as np, pandas as pd, cv2, torch
from pathlib import Path

def find(name, must):
    pats = [f'/content/drive/MyDrive/{name}', f'/content/drive/MyDrive/*/{name}', f'/content/drive/MyDrive/*/*/{name}',
            f'/content/drive/.shortcut-targets-by-id/*/{name}', f'/content/drive/.shortcut-targets-by-id/*/*/{name}']
    c = [p for pat in pats for p in glob.glob(pat) if os.path.exists(os.path.join(p, must))]
    assert c, f'{name} not found (needs {must})'
    return Path(c[0])

ROOT = find('AI251P_Capstone_CrossDomainAerial', '03_Detection')
Z = find('Ziyad_folder', 'yolo11n_seg_1280-3/weights/last.pt')
MANIFEST = ROOT / '04_Integration/2026-11-16_Pipeline_v1/manifest.json'
SERVICE = ROOT / '04_Integration/Service/2026-11-16_Service_v1'
CLS_DIR = ROOT / '02_Classification/Export/2026-10-31_ClsResNet18_s43_v1'
DET = ROOT / '03_Detection/2026-11-04_P08_Baseline/YOLO11n_Joint_StrongerAug/weights/best.pt'
SEG = Z / 'yolo11n_seg_1280-3/weights/last.pt'
P30 = ROOT / '03_Detection/2026-11-06_P09_Evaluation/P30_OperatingPoint/P30_OperatingPoint_v1.json'
VAL_IMG, VAL_LBL = Z / 'yolo_dataset/images/val', Z / 'yolo_dataset/labels/val'
CLIP, EXAMPLE_JPG = SERVICE / 'example_clip.mp4', SERVICE / 'example_AlHada_CLIP01_t0150.jpg'
OUT = ROOT / '04_Integration/2026-11-18_P31_Closeout'; OUT.mkdir(exist_ok=True)
for p in [MANIFEST, SERVICE / 'pipeline.py', SERVICE / 'app.py', CLS_DIR / 'classify.py', CLS_DIR / 'model.onnx', DET, SEG, P30, VAL_IMG, VAL_LBL, CLIP, EXAMPLE_JPG]:
    assert p.exists(), p
assert 'test' not in str(VAL_IMG).split('/')[-1]
# Pipeline_v2 release: manifest with sha256 + validation operating point, launcher serve.py, segmentation weights copied in
REL = ROOT / '04_Integration/2026-11-18_Pipeline_v2'; (REL / 'models').mkdir(parents=True, exist_ok=True)
SERVE_PY = r'''# AI251P service launcher: builds the pipeline ONLY from a manifest, checks the model files and one image
# before it starts, then serves Rayan's app (app.create_app). If any check fails, the service does not start.
#
#     app = make_app('manifest.json', root='/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial',
#                    domain='urban', out_dir='results', check_image='frame.jpg')
#
# The manifest gives, per stage: checkpoint path (relative to root, or absolute), expected sha256, thresholds, input size.
# domain picks the detector threshold chosen on validation (P30): 'sea' or 'urban'.
import hashlib, json, time
from pathlib import Path
import numpy as np, cv2


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def _path(root, p):
    p = Path(p)
    return p if p.is_absolute() else Path(root) / p


def load_manifest(manifest_path):
    m = json.loads(Path(manifest_path).read_text())
    for stage in ('detector', 'classifier', 'segmentation'):
        if stage not in m['stages']:
            raise ValueError(f'manifest has no {stage} stage')
        if not m['stages'][stage].get('sha256'):
            raise ValueError(f'manifest has no sha256 for {stage}')
    return m


def verify_models(m, root):
    """sha256 of every model file against the manifest. Returns a list of checks; raises if one fails."""
    checks = []
    for stage in ('detector', 'classifier', 'segmentation'):
        s = m['stages'][stage]
        p = _path(root, s['checkpoint'])
        if not p.is_file():
            raise FileNotFoundError(f'{stage}: {p} not found')
        got = sha256(p)
        checks.append({'check': f'{stage} sha256', 'expected': s['sha256'], 'actual': got, 'pass': got == s['sha256']})
    bad = [c['check'] for c in checks if not c['pass']]
    if bad:
        raise RuntimeError(f'model files differ from the manifest: {bad}')
    return checks


def build_pipeline(m, root, domain):
    import pipeline as pl
    d, c, s = m['stages']['detector'], m['stages']['classifier'], m['stages']['segmentation']
    if domain not in d['conf_by_domain']:
        raise ValueError(f"domain must be one of {list(d['conf_by_domain'])}")
    pipe = pl.Pipeline(_path(root, d['checkpoint']), _path(root, c['checkpoint']),
                       det_conf=d['conf_by_domain'][domain], det_iou=d['nms_iou'], det_imgsz=d['imgsz'],
                       seg_weights=_path(root, s['checkpoint']), seg_backend=s['backend'], seg_conf=s['conf'],
                       seg_imgsz=s['imgsz'], seg_match_iou=s['match_iou'])
    names = [pipe.det_names[i] for i in sorted(pipe.det_names)]
    if names != d['classes']:
        raise RuntimeError(f'detector classes {names} differ from the manifest {d["classes"]}')
    return pipe


def startup_check(pipe, image_path):
    """Runs one image; every object must carry class, confidence, box and size. Raises if not."""
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise RuntimeError(f'start-up image {image_path} cannot be read')
    t = time.perf_counter()
    r = pipe.analyze(np.ascontiguousarray(bgr[..., ::-1]))
    ms = (time.perf_counter() - t) * 1e3
    bad = [o for o in r['objects'] if not (o.get('class') and 0 <= o.get('confidence', -1) <= 1
                                            and len(o.get('box_xywh', [])) == 4 and o.get('size'))]
    if bad:
        raise RuntimeError(f'start-up image: {len(bad)} objects without class, confidence, box or size')
    return {'check': 'start-up image', 'objects': len(r['objects']),
            'with_mask': sum(o['mask'] is not None for o in r['objects']), 'ms': round(ms, 1), 'pass': True}


def make_app(manifest_path, root, domain, out_dir, check_image, **app_kw):
    import pipeline as pl, app as svc
    m = load_manifest(manifest_path)
    checks = verify_models(m, root)
    pipe = build_pipeline(m, root, domain)
    checks.append(startup_check(pipe, check_image))
    info = {'manifest': str(manifest_path), 'domain': domain,
            'detector_conf': m['stages']['detector']['conf_by_domain'][domain],
            'detector_conf_source': m['stages']['detector'].get('conf_source'),
            'segmentation_conf': m['stages']['segmentation']['conf'],
            'startup_checks': [{k: v for k, v in c.items() if k in ('check', 'pass', 'objects', 'ms')} for c in checks]}
    return svc.create_app(pipe, pl.draw, out_dir, info=info, **app_kw), pipe, checks
'''
MANIFEST_V2 = r'''{
 "created": "2026-11-18",
 "version": "Pipeline_v2",
 "based_on": "04_Integration/2026-11-16_Pipeline_v1/manifest.json (Rayan Alsulami); v2 adds sha256 per model, the validation operating point per domain, and the segmentation weights copied into the release",
 "root": "AI251P_Capstone_CrossDomainAerial (paths below are relative to it)",
 "stages": {
  "detector": {
   "checkpoint": "03_Detection/2026-11-04_P08_Baseline/YOLO11n_Joint_StrongerAug/weights/best.pt",
   "sha256": "12089000a5be6b30df3ab2535e3566ba28ce9b2eb799e42f0e166c8612548582",
   "conf_by_domain": {
    "sea": 0.2,
    "urban": 0.05
   },
   "conf_source": "P30: max F2 on validation, grid 0.05-0.95 (03_Detection/2026-11-06_P09_Evaluation/P30_OperatingPoint/P30_OperatingPoint_v1.json)",
   "nms_iou": 0.7,
   "imgsz": 640,
   "classes": [
    "swimmer",
    "boat",
    "jetski",
    "life_saving_appliance",
    "buoy",
    "car",
    "truck",
    "bus"
   ]
  },
  "classifier": {
   "checkpoint": "02_Classification/Export/2026-10-31_ClsResNet18_s43_v1/model.onnx",
   "sha256": "7a82f8742aad3db7848bfc9740735262fac2c7b0c533d16689d67e1493a42b8b",
   "classes": [
    "swimmer",
    "boat",
    "jetski",
    "life_saving_appliances",
    "buoy",
    "car",
    "truck",
    "bus"
   ],
   "input": "64 px square crop, raw logits, argmax"
  },
  "segmentation": {
   "checkpoint": "04_Integration/2026-11-18_Pipeline_v2/models/seg_last.pt",
   "copied_from": "Ziyad_folder/yolo11n_seg_1280-3/weights/last.pt",
   "sha256": "264be4d7974ee7e5830d095c15312b78ea97410def49ef17817aa941db863953",
   "backend": "yolo-seg",
   "conf": 0.25,
   "imgsz": 1280,
   "match_iou": 0.3
  }
 },
 "size": {
  "altitude_m": null,
  "hfov_deg": null,
  "from": "mask (area, minimum-area rectangle) else box"
 }
}'''
(REL / 'serve.py').write_text(SERVE_PY); (REL / 'manifest.json').write_text(MANIFEST_V2)
import shutil
if not (REL / 'models/seg_last.pt').exists(): shutil.copy2(SEG, REL / 'models/seg_last.pt')
sys.path[:0] = [str(REL), str(SERVICE), str(CLS_DIR)]
import classify as clf, pipeline as pl, app as svc, serve
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
HAS_T4 = 'T4' in GPU
print('device:', GPU, '| benchmark', 'ON' if HAS_T4 else 'SKIPPED (needs a T4)')
assert HAS_T4 or not ONLY_BENCHMARK, 'ONLY_BENCHMARK needs a T4 GPU'
import ultralytics, onnxruntime
VERSIONS = {'python': sys.version.split()[0], 'torch': torch.__version__, 'ultralytics': ultralytics.__version__,
            'onnxruntime': onnxruntime.__version__, 'opencv': cv2.__version__, 'gpu': GPU}
print('ROOT', ROOT); print('Ziyad_folder', Z); print(VERSIONS)

In [ ]:
# checksums
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
SHA = {'detector_best_pt': sha256(DET), 'classifier_model_onnx': sha256(CLS_DIR / 'model.onnx'), 'segmentation_last_pt': sha256(SEG)}
EXPECTED_SEG = '264be4d7974ee7e5830d095c15312b78ea97410def49ef17817aa941db863953'   # Drive file of 17 Nov, 6,112,164 bytes
print(json.dumps(SHA, indent=1))
print('segmentation checksum matches the 17 Nov file:', SHA['segmentation_last_pt'] == EXPECTED_SEG)
json.dump({'sha256': SHA, 'paths': {'detector': str(DET.relative_to(ROOT)), 'classifier': str((CLS_DIR / 'model.onnx').relative_to(ROOT)),
           'segmentation': 'Ziyad_folder/yolo11n_seg_1280-3/weights/last.pt'}, 'versions': VERSIONS},
          open(OUT / 'P31_Checksums_v1.json', 'w'), indent=1)

In [ ]:
# pipeline from the manifest
M = json.load(open(MANIFEST))['stages']
OP = json.load(open(P30))['chosen']                     # validation operating point (P30): sea 0.20, urban 0.05
print('manifest detector', M['detector']['conf'], M['detector']['nms_iou'], M['detector']['imgsz'], '| segmentation', M['segmentation'])
print('P30 operating point', OP)
def build(det_conf):
    return pl.Pipeline(DET, CLS_DIR / 'model.onnx', det_conf=det_conf, det_iou=M['detector']['nms_iou'], det_imgsz=M['detector']['imgsz'],
                       seg_weights=SEG, seg_backend=M['segmentation']['backend'], seg_conf=M['segmentation']['conf'],
                       seg_imgsz=M['segmentation']['imgsz'], seg_match_iou=M['segmentation']['match_iou'])
# Al-Hada is a road scene, so the urban operating point applies.
M2 = serve.load_manifest(REL / 'manifest.json')
assert M2['stages']['detector']['conf_by_domain'] == {k: float(v) for k, v in OP.items()}, 'manifest v2 thresholds differ from P30'
PIPES = {'manifest (det conf 0.25)': build(M['detector']['conf']),
         f"operating point (det conf {OP['urban']})": serve.build_pipeline(M2, ROOT, 'urban')}   # built from manifest v2 only
print('built', list(PIPES))

In [ ]:
# benchmark
if HAS_T4:
    exec(compile(r'''
names = sorted(p.name for p in VAL_IMG.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
random.Random(42).shuffle(names); names = sorted(names[:30])
FRAMES = [np.ascontiguousarray(cv2.imread(str(VAL_IMG / n))[..., ::-1]) for n in names]   # decoded once: decode is excluded
WARMUP, PASSES = 10, 3

def gpu_used_mb():
    o = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout
    return float(o.strip().splitlines()[0])

ROWS, SUMMARY = [], {}
for label, pipe in PIPES.items():
    for i in range(WARMUP): pipe.analyze(FRAMES[i % len(FRAMES)])
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    peak = {'gpu_mb': gpu_used_mb()}; stop = threading.Event()
    def poll():
        while not stop.is_set(): peak['gpu_mb'] = max(peak['gpu_mb'], gpu_used_mb()); time.sleep(0.2)
    th = threading.Thread(target=poll); th.start()
    t0 = time.perf_counter(); n = 0
    for p in range(PASSES):
        for k, f in enumerate(FRAMES):
            r = pipe.analyze(f); n += 1
            ROWS.append({'setting': label, 'pass': p, 'frame': names[k], 'n_objects': len(r['objects']), **{f'{s}_ms': v for s, v in r['timing_ms'].items()}})
    torch.cuda.synchronize(); wall = time.perf_counter() - t0; stop.set(); th.join()
    df = pd.DataFrame([x for x in ROWS if x['setting'] == label])
    SUMMARY[label] = {'timed_calls': n, 'warmup_calls': WARMUP, 'frames': len(FRAMES), 'wall_s': round(wall, 2),
                      'throughput_fps': round(n / wall, 2), 'objects_per_frame_mean': round(df.n_objects.mean(), 1),
                      'gpu_mem_used_peak_mb_nvidia_smi': peak['gpu_mb'],
                      'torch_max_allocated_mb': round(torch.cuda.max_memory_allocated() / 2**20, 1),
                      'torch_max_reserved_mb': round(torch.cuda.max_memory_reserved() / 2**20, 1),
                      'process_peak_rss_mb': round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1),
                      'latency_ms': {s: {q: round(float(np.percentile(df[f'{s}_ms'], v)), 1) for q, v in (('p50', 50), ('p95', 95), ('p99', 99))}
                                     | {'mean': round(float(df[f'{s}_ms'].mean()), 1)} for s in ('detect', 'classify', 'segment', 'size', 'total')}}
pd.DataFrame(ROWS).to_csv(OUT / 'P31_BenchmarkCalls_v1.csv', index=False)
json.dump({'conditions': {'frames': f'{len(FRAMES)} Al-Hada validation frames (CLIP01, Ziyad_folder/yolo_dataset/images/val, seed 42)',
                          'batch': 1, 'decode': 'excluded (frames decoded before timing)', 'warmup_calls': WARMUP, 'passes': PASSES,
                          'versions': VERSIONS, 'created_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M')},
           'settings': SUMMARY}, open(OUT / 'P31_BenchmarkSummary_v1.json', 'w'), indent=1)
for k, v in SUMMARY.items():
    print(f'\n== {k}: {v["timed_calls"]} timed calls after {v["warmup_calls"]} warm-up; throughput {v["throughput_fps"]} frames/s; '
          f'{v["objects_per_frame_mean"]} objects/frame')
    print(f'   GPU memory used (nvidia-smi peak) {v["gpu_mem_used_peak_mb_nvidia_smi"]:.0f} MB; torch max allocated {v["torch_max_allocated_mb"]} MB; '
          f'process peak RSS {v["process_peak_rss_mb"]} MB')
    print(pd.DataFrame(v['latency_ms']).T.to_string())

import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
B = pd.DataFrame(ROWS); stages = ['detect', 'classify', 'segment', 'size', 'total']
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={'width_ratios': [3, 1]})
for i, (label, g) in enumerate(B.groupby('setting', sort=False)):
    pos = np.arange(len(stages)) + (i - 0.5) * 0.36
    bp = axes[0].boxplot([g[f'{s}_ms'] for s in stages], positions=pos, widths=0.32, patch_artist=True, showfliers=False)
    for b in bp['boxes']: b.set_facecolor(['#9db7d5', '#e3a87a'][i])
    axes[0].plot([], [], 's', color=['#9db7d5', '#e3a87a'][i], label=label)
axes[0].axhline(200, ls='--', c='k', lw=0.8); axes[0].text(4.45, 205, '200 ms budget', ha='right', fontsize=8)
axes[0].set_xticks(range(len(stages)), stages); axes[0].set_ylabel('ms per frame'); axes[0].legend(fontsize=8, loc='upper left')
axes[1].bar(range(len(SUMMARY)), [v['throughput_fps'] for v in SUMMARY.values()], color=['#9db7d5', '#e3a87a'])
axes[1].set_xticks(range(len(SUMMARY)), ['conf 0.25', f"conf {OP['urban']}"]); axes[1].set_ylabel('frames per second')
fig.suptitle(f'Pipeline on a {GPU}, batch 1, {len(FRAMES)} Al-Hada val frames x {PASSES}, after {WARMUP} warm-up calls', fontsize=9)
fig.tight_layout(); fig.savefig(OUT / 'P31_benchmark.png', dpi=200); plt.show()

''', 'benchmark', 'exec'))
else:
    print('benchmark skipped: no T4 GPU in this session')

In [ ]:
# video run at the operating point
if ONLY_BENCHMARK: raise SystemExit('ONLY_BENCHMARK: stop here')
pipe = PIPES[f"operating point (det conf {OP['urban']})"]
cap = cv2.VideoCapture(str(CLIP)); fps = cap.get(cv2.CAP_PROP_FPS); W, H = int(cap.get(3)), int(cap.get(4))
writer = cv2.VideoWriter(str(OUT / 'P31_video_annotated.mp4'), cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
VID, k, t_start = [], 0, time.perf_counter()
while True:
    ok, bgr = cap.read()
    if not ok: break
    t = time.perf_counter(); rgb = np.ascontiguousarray(bgr[..., ::-1]); dec = (time.perf_counter() - t) * 1e3
    r = pipe.analyze(rgb); vis = pl.draw(rgb, r); writer.write(vis[..., ::-1])
    VID.append({'frame': k, 'n_objects': len(r['objects']), 'with_mask': sum(o['mask'] is not None for o in r['objects']),
                'convert_ms': dec, **{f'{s}_ms': v for s, v in r['timing_ms'].items()}})
    if k in (0, 10, 19): cv2.imwrite(str(OUT / f'P31_video_frame{k:02d}.jpg'), vis[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, 85])
    k += 1
wall = time.perf_counter() - t_start; cap.release(); writer.release()
V = pd.DataFrame(VID); V.to_csv(OUT / 'P31_VideoFrames_v1.csv', index=False)
VS = {'device': GPU, 'clip': 'Service/2026-11-16_Service_v1/example_clip.mp4 (Al-Hada)', 'source_fps': fps, 'size': [W, H], 'frames': len(V),
      'every_frame': True, 'det_conf': OP['urban'], 'objects_per_frame_mean': round(V.n_objects.mean(), 1),
      'objects_with_mask': int(V.with_mask.sum()), 'objects_total': int(V.n_objects.sum()),
      'pipeline_ms_mean': round(V.total_ms.mean(), 1), 'pipeline_ms_p95': round(float(np.percentile(V.total_ms, 95)), 1),
      'processing_fps_pipeline_only': round(1000 / V.total_ms.mean(), 2), 'processing_fps_end_to_end_incl_read_draw_write': round(len(V) / wall, 2)}
json.dump(VS, open(OUT / 'P31_VideoSummary_v1.json', 'w'), indent=1)
print(json.dumps(VS, indent=1))

In [ ]:
# HTTP service: start, health, analyze, confirm
if ONLY_BENCHMARK: raise SystemExit('ONLY_BENCHMARK: stop here')
import uvicorn, requests
# 1) a manifest with a wrong checksum must stop the service before it starts
bad = json.loads((REL / 'manifest.json').read_text()); bad['stages']['segmentation']['sha256'] = '0' * 64
(OUT / 'manifest_bad_checksum.json').write_text(json.dumps(bad))
try:
    serve.make_app(OUT / 'manifest_bad_checksum.json', ROOT, 'urban', OUT / 'service_results', EXAMPLE_JPG); print('ERROR: bad manifest was accepted')
except RuntimeError as e:
    print('start-up refused a wrong checksum, as intended ->', e)
# 2) the real start: manifest v2 only -> sha256 of every model -> one start-up image -> serve
api, pipe_srv, CHECKS = serve.make_app(REL / 'manifest.json', ROOT, 'urban', OUT / 'service_results', EXAMPLE_JPG)
for c in CHECKS: print('start-up check:', c['check'], '->', 'PASS' if c['pass'] else 'FAIL', {k: v for k, v in c.items() if k in ('objects', 'with_mask', 'ms')})
json.dump(CHECKS, open(OUT / 'P31_StartupChecks_v1.json', 'w'), indent=1)
server = uvicorn.Server(uvicorn.Config(api, host='127.0.0.1', port=8000, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:8000/health', timeout=1).ok: break
    except Exception: time.sleep(0.5)
print('GET /health ->', json.dumps(requests.get('http://127.0.0.1:8000/health').json(), indent=1)[:900])
r = requests.post('http://127.0.0.1:8000/analyze', files={'file': open(EXAMPLE_JPG, 'rb')}, params={'top_k': 10}).json()
print('\nPOST /analyze ->', r['request_id'], '|', r['n_candidates'], 'candidates |', r['timing'])
for c in r['candidates'][:3]:
    print(f"  #{c['rank']} {c['candidate_id']} {c['class']} conf {c['confidence']:.3f} box {c['box_xywh']} "
          f"length {c['size']['length_px']:.0f} px (from {c['size']['source']}) -> {c['status']}")
cf = requests.post(f"http://127.0.0.1:8000/confirm/{r['request_id']}", json={'candidate_id': 'c001', 'decision': 'confirmed', 'operator': 'P31 test'}).json()
print('\nPOST /confirm ->', json.dumps(cf)[:400])
img = requests.get(f"http://127.0.0.1:8000{r['annotated_image_url']}").content
open(OUT / 'P31_service_annotated.jpg', 'wb').write(img)
from IPython.display import Image as _Img, display; display(_Img(data=img, width=900))
server.should_exit = True

In [ ]:
# segmentation threshold on validation
if ONLY_BENCHMARK: raise SystemExit('ONLY_BENCHMARK: stop here')
# Rule fixed before running: confidence on the grid 0.05..0.95 (step 0.05) that maximises mask F1 on VALIDATION,
# ties -> lower. Match: one-to-one, class-consistent, mask IoU >= 0.5, predictions in descending confidence. Test is not used.
from ultralytics import YOLO
seg = YOLO(str(SEG)); GRIDS = np.round(np.arange(0.05, 0.951, 0.05), 2)
def gt_polys(lbl, w, h):
    out = []
    if lbl.exists():
        for line in lbl.read_text().split('\n'):
            v = line.split()
            if len(v) >= 7: out.append((int(v[0]), np.array(v[1:], float).reshape(-1, 2) * [w, h]))
    return out
def mask_iou(pa, pb):
    a, b = np.asarray(pa), np.asarray(pb)
    if len(a) < 3 or len(b) < 3: return 0.0
    x0, y0 = np.floor(np.minimum(a.min(0), b.min(0))).astype(int); x1, y1 = np.ceil(np.maximum(a.max(0), b.max(0))).astype(int) + 1
    ma = np.zeros((y1 - y0, x1 - x0), np.uint8); mb = ma.copy()
    cv2.fillPoly(ma, [np.round(a - [x0, y0]).astype(np.int32)], 1); cv2.fillPoly(mb, [np.round(b - [x0, y0]).astype(np.int32)], 1)
    u = np.logical_or(ma, mb).sum(); return float(np.logical_and(ma, mb).sum() / u) if u else 0.0
def box(p): p = np.asarray(p); return np.r_[p.min(0), p.max(0)] if len(p) else np.zeros(4)
IMGS = sorted(p for p in VAL_IMG.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')); CACHE = []
for f in IMGS:
    im = cv2.imread(str(f)); h, w = im.shape[:2]
    r = seg.predict(im, conf=0.001, imgsz=M['segmentation']['imgsz'], max_det=300, verbose=False)[0]
    gts = gt_polys(VAL_LBL / (f.stem + '.txt'), w, h)
    P = [] if r.masks is None else list(zip(r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy(), r.masks.xy))
    P.sort(key=lambda t: -t[1])
    gb = [box(g[1]) for g in gts]; I = np.zeros((len(P), len(gts)))
    for i, (c, s, poly) in enumerate(P):
        pb = box(poly)
        for j, (gc, gp) in enumerate(gts):
            if gc == c and pb[0] < gb[j][2] and gb[j][0] < pb[2] and pb[1] < gb[j][3] and gb[j][1] < pb[3]: I[i, j] = mask_iou(poly, gp)
    CACHE.append((np.array([p[1] for p in P]), I, len(gts)))
print('val images', len(IMGS), '| objects', sum(c[2] for c in CACHE))
def score(thr):
    TP = FP = FN = 0
    for conf, I, ng in CACHE:
        used = np.zeros(ng, bool); keep = np.flatnonzero(conf >= thr)
        for i in keep:
            cand = np.flatnonzero(~used & (I[i] >= 0.5))
            if len(cand): used[cand[np.argmax(I[i, cand])]] = True; TP += 1
            else: FP += 1
        FN += int((~used).sum())
    p = TP / max(TP + FP, 1); r = TP / max(TP + FN, 1)
    return {'thr': float(thr), 'tp': TP, 'fp': FP, 'fn': FN, 'precision': p, 'recall': r, 'f1': 2 * p * r / max(p + r, 1e-9)}
SS = pd.DataFrame([score(t) for t in GRIDS])
best = SS.loc[SS.assign(k=SS.f1.round(6)).sort_values(['k', 'thr'], ascending=[False, True]).index[0]]
SS.to_csv(OUT / 'P31_SegValSweep_v1.csv', index=False)
json.dump({'device': GPU, 'rule': 'max mask F1 on val (class-consistent, mask IoU >= 0.5), grid 0.05..0.95 step 0.05, ties -> lower',
           'imgsz': M['segmentation']['imgsz'], 'chosen_conf': float(best.thr), 'pipeline_conf': M['segmentation']['conf'],
           'val_images': len(IMGS), 'created_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M')},
          open(OUT / 'P31_SegOperatingPoint_v1.json', 'w'), indent=1)
print(SS.round(3).to_string(index=False))
print(f"\nchosen on val: seg conf = {best.thr:.2f} (P {best.precision:.3f}, R {best.recall:.3f}, F1 {best.f1:.3f}); "
      f"pipeline uses {M['segmentation']['conf']}: F1 {SS.loc[np.isclose(SS.thr, M['segmentation']['conf']), 'f1'].item():.3f}")
print('\nP31 done ->', OUT)